# Notebook 2: GEDI UOI Signal Generation (GEE)

Two-phase pipeline:
- **Phase 1**: Export raw GEDI UOI + forest/topo mask at 1km (separate assets)
- **Phase 2**: Combine assets into masked GEDI product

Set `PHASE` in Block 1 to control which step runs.

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")

# Pipeline Phase:
#   1 = Export raw GEDI + mask at 1km (run first)
#   2 = Combine into masked GEDI product (run after Phase 1 completes)
PHASE = 1

ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]
EXPORT_SCALE = 1000

# Datasets
GEDI_L2B = 'LARSE/GEDI/GEDI02_B_002_MONTHLY'
FOREST_MASK = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'
FOREST_CLASS = 10
FOREST_COVER_THRESHOLD = 0.95
SRTM = 'USGS/SRTMGL1_003'
START_DATE = '2020-01-01'
END_DATE = '2023-12-31'

print(f"\u2713 Configuration loaded. PHASE = {PHASE}")

In [ ]:
# =============================================================================
# BLOCK 2: METHODOLOGICAL LOGIC (FUNCTIONS)
# =============================================================================

# ---- Phase 1: Build intermediate assets ----

def build_gedi_raw():
    """Raw GEDI UOI + N at 1km. No masks. No reproject.
    The export's scale parameter defines the output grid.
    """
    gedi = ee.ImageCollection(GEDI_L2B).filterDate(START_DATE, END_DATE)
    
    def calc_uoi(img):
        pai = img.select('pai')
        pavd_z0 = img.select('pavd_z0')
        uoi = ee.Image(1).subtract(pavd_z0.divide(pai))
        return uoi.clamp(0, 1).rename('UOI')
    
    uoi_col = gedi.map(calc_uoi)
    mean_uoi = uoi_col.mean().rename('UOI_mean')
    count_n = uoi_col.count().rename('N')
    native_proj = gedi.first().projection()
    
    # reduceResolution aggregates 25m->1km (~1600 pixels per cell)
    # No reproject — export's scale defines the output grid
    agg_uoi = mean_uoi.setDefaultProjection(native_proj).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535)
    agg_n = count_n.setDefaultProjection(native_proj).reduceResolution(
        reducer=ee.Reducer.sum(), maxPixels=65535)
    
    return ee.Image.cat([agg_uoi, agg_n])

def build_mask():
    """Forest fraction + topo fraction at 1km.
    Exported separately so NB3 can apply harmonized masking.
    """
    # JRC TMF forest fraction
    tmf_col = ee.ImageCollection(FOREST_MASK)
    tmf_proj = tmf_col.first().projection()
    forest = tmf_col.mosaic().eq(FOREST_CLASS).setDefaultProjection(tmf_proj)
    forest_frac = forest.reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).rename('forest_fraction')
    
    # SRTM topo: fraction of pixels with elev<1000 AND slope<10
    srtm = ee.Image(SRTM)
    topo = srtm.select('elevation').lt(1000).And(ee.Terrain.slope(srtm).lt(10))
    topo_frac = topo.reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).rename('topo_fraction')
    
    return ee.Image.cat([forest_frac, topo_frac])

# ---- Phase 2: Combine assets ----

def build_masked_gedi(basin_name):
    """Loads Phase 1 assets and applies forest+topo mask."""
    gedi = ee.Image(f'{ASSET_ROOT}/Openness_raw/GEDI_raw_{basin_name}')
    mask = ee.Image(f'{ASSET_ROOT}/Openness_raw/Mask_{basin_name}')
    
    return gedi.updateMask(
        mask.select('forest_fraction').gte(FOREST_COVER_THRESHOLD)
    ).updateMask(
        mask.select('topo_fraction').gte(0.5)
    )

print("\u2713 Functions loaded.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    if PHASE == 1:
        print("Phase 1 tests...")
        gedi = build_gedi_raw()
        bands = gedi.bandNames().getInfo()
        assert 'UOI_mean' in bands and 'N' in bands, f"Got: {bands}"
        print(f"  \u2713 GEDI raw bands: {bands}")
        
        mask = build_mask()
        mbands = mask.bandNames().getInfo()
        assert 'forest_fraction' in mbands and 'topo_fraction' in mbands, f"Got: {mbands}"
        print(f"  \u2713 Mask bands: {mbands}")
        print("\n  \u2713 Phase 1 tests passed. Export, then set PHASE=2.")
    
    elif PHASE == 2:
        print("Phase 2 tests...")
        try:
            masked = build_masked_gedi('Congo')
            bands = masked.bandNames().getInfo()
            assert 'UOI_mean' in bands and 'N' in bands, f"Got: {bands}"
            print(f"  \u2713 Masked GEDI bands: {bands}")
            print("\n  \u2713 Phase 2 tests passed. Ready to export.")
        except Exception as e:
            print(f"  \u2717 Error: {e}")
            print("  Have Phase 1 assets finished exporting?")

run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 4: EXECUTION (ASSET EXPORT)
# =============================================================================

def safe_start(task, asset_id):
    try:
        ee.data.deleteAsset(asset_id)
        print(f"    Deleted existing: {asset_id.split('/')[-1]}")
    except Exception:
        pass
    task.start()

def export_phase_1(dry_run=True):
    gedi = build_gedi_raw()
    mask = build_mask()
    tasks = []
    for basin_name, basin_geom in BASINS:
        for img, label in [(gedi, 'GEDI_raw'), (mask, 'Mask')]:
            aid = f'{ASSET_ROOT}/Openness_raw/{label}_{basin_name}'
            t = ee.batch.Export.image.toAsset(
                image=img, description=f'{label}_{basin_name}',
                assetId=aid, region=basin_geom,
                scale=EXPORT_SCALE, crs='EPSG:4326', maxPixels=1e13)
            tasks.append((t, aid))
    print(f"\u2713 Phase 1: {len(tasks)} tasks configured.")
    if dry_run:
        print("DRY RUN. Call export_phase_1(dry_run=False) to start.")
    else:
        for t, aid in tasks:
            safe_start(t, aid)
        print("\u2713 Started! Wait for completion, then set PHASE=2.")

def export_phase_2(dry_run=True):
    tasks = []
    for basin_name, basin_geom in BASINS:
        masked = build_masked_gedi(basin_name)
        aid = f'{ASSET_ROOT}/Openness_raw/GEDI_1km_{basin_name}'
        t = ee.batch.Export.image.toAsset(
            image=masked, description=f'GEDI_1km_{basin_name}',
            assetId=aid, region=basin_geom,
            scale=EXPORT_SCALE, crs='EPSG:4326', maxPixels=1e13)
        tasks.append((t, aid))
    print(f"\u2713 Phase 2: {len(tasks)} tasks configured.")
    if dry_run:
        print("DRY RUN. Call export_phase_2(dry_run=False) to start.")
    else:
        for t, aid in tasks:
            safe_start(t, aid)
        print("\u2713 Started! Monitor at https://code.earthengine.google.com/tasks")

if PHASE == 1:
    export_phase_1(dry_run=True)
elif PHASE == 2:
    export_phase_2(dry_run=True)